# Training Curves — NYUv2 Depth LoRA (50 epochs)
Parses `log.txt` in real-time and plots loss + learning rate curves.
Re-run any cell to refresh while training is still running.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

LOG_PATH = '../output/depth/nyu_rgb_r8/log.txt'

In [ ]:
# --- Parse log.txt ---
# Two sources:
#   1. Per-step lines: 'Epoch [N]  [STEP/TOTAL] ... lr: X  loss: CUR (AVG)'
#   2. Epoch summary:  '[Train] Averaged stats: lr: X  loss: CUR (AVG)'

step_re  = re.compile(
    r'Epoch \[(\d+)\]\s+\[\s*(\d+)/\d+\].*?lr:\s*([0-9e.+-]+)'
    r'.*?loss:\s*[0-9e.+-]+\s*\(([0-9e.+-]+)\)'
    r'.*?loss_tok_depth@224:\s*[0-9e.+-]+\s*\(([0-9e.+-]+)\)'
)
epoch_re = re.compile(
    r'\[Train\] Averaged stats:.*?lr:\s*([0-9e.+-]+)'
    r'.*?loss:\s*[0-9e.+-]+\s*\(([0-9e.+-]+)\)'
    r'.*?loss_tok_depth@224:\s*[0-9e.+-]+\s*\(([0-9e.+-]+)\)'
)
val_re   = re.compile(
    r'\[Val\].*?loss:\s*[0-9e.+-]+\s*\(([0-9e.+-]+)\)'
)

step_rows, epoch_rows, val_rows = [], [], []
current_epoch = 0

with open(LOG_PATH) as f:
    for line in f:
        m = step_re.search(line)
        if m:
            ep, step, lr, loss_avg, depth_avg = m.groups()
            step_rows.append(dict(
                epoch=int(ep), step=int(step),
                lr=float(lr), loss=float(loss_avg),
                depth_loss=float(depth_avg)
            ))
            current_epoch = int(ep)
            continue
        m = epoch_re.search(line)
        if m:
            lr, loss_avg, depth_avg = m.groups()
            epoch_rows.append(dict(
                epoch=current_epoch,
                lr=float(lr), loss=float(loss_avg),
                depth_loss=float(depth_avg)
            ))
            continue
        m = val_re.search(line)
        if m:
            val_rows.append(dict(epoch=current_epoch, val_loss=float(m.group(1))))

steps  = pd.DataFrame(step_rows)
epochs = pd.DataFrame(epoch_rows)
vals   = pd.DataFrame(val_rows)

print(f"Steps parsed : {len(steps)}")
print(f"Epochs parsed: {len(epochs)}  (completed)")
print(f"Val points   : {len(vals)}")
if not epochs.empty:
    print(f"Last epoch   : {int(epochs.epoch.max())}  |  loss={epochs.loss.iloc[-1]:.4f}  |  lr={epochs.lr.iloc[-1]:.2e}")

In [ ]:
# --- Plot: Loss curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NYUv2 Depth — LoRA fine-tuning (rank=8, XL)', fontsize=13)

# Left: step-level loss (smoothed) + epoch averages
ax = axes[0]
if not steps.empty:
    # global step index
    steps_per_epoch = steps.groupby('epoch')['step'].max().mean()
    steps['global_step'] = steps['epoch'] * steps_per_epoch + steps['step']
    # thin out for performance: plot every 50th step
    thin = steps.iloc[::50]
    ax.plot(thin['global_step'], thin['loss'], alpha=0.35, color='steelblue', lw=0.8, label='step loss (avg)')

if not epochs.empty:
    # epoch midpoint in global steps
    epochs['global_step'] = (epochs['epoch'] + 0.5) * steps_per_epoch
    ax.plot(epochs['global_step'], epochs['loss'], 'o-', color='steelblue', lw=2, ms=5, label='epoch avg loss')

if not vals.empty:
    vals['global_step'] = (vals['epoch'] + 1) * steps_per_epoch
    ax.plot(vals['global_step'], vals['val_loss'], 's--', color='tomato', lw=2, ms=6, label='val loss')

ax.set_xlabel('Global step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: same but by epoch
ax2 = axes[1]
if not epochs.empty:
    ax2.plot(epochs['epoch'], epochs['loss'], 'o-', color='steelblue', lw=2, ms=5, label='train loss')
if not vals.empty:
    ax2.plot(vals['epoch'], vals['val_loss'], 's--', color='tomato', lw=2, ms=6, label='val loss')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-entropy loss')
ax2.set_title('Loss by epoch')
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/loss_curves.png')

In [ ]:
# --- Plot: Learning rate schedule ---
fig, ax = plt.subplots(figsize=(9, 4))

if not steps.empty:
    thin = steps.iloc[::50]
    ax.plot(thin['global_step'], thin['lr'], color='darkorange', lw=1.2)

ax.set_xlabel('Global step')
ax.set_ylabel('Learning rate')
ax.set_title('Learning rate schedule (cosine + warmup)')
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/lr_schedule.png')